In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    classification_report,
)

pd.set_option("display.max_columns", None)

In [2]:
CSV_PATH = "../../data/forecast_vs_actual.csv"
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
df.head()

Shape: (4800, 23)


,forecast_id,station_id,target_timestamp,lead_time_hours,forecast_issued_at,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_condition,forecasted_umbrella,forecasted_clothing,actual_temp_c,actual_precip_mm,actual_wind_kmh,actual_condition,actual_umbrella,actual_clothing,temp_error_c,precip_error_mm,wind_error_kmh,umbrella_correct,clothing_correct,condition_correct
0,FCST-000001,WS-001,2025-03-14 00:00:00,3,2025-03-13 21:00:00,-11.3,0.0,12.6,clear,0,heavy_winter_coat_gloves_hat,-12.1,0.0,12.0,clear,0,heavy_winter_coat_gloves_hat,0.8,0.0,0.6,1,1,1
1,FCST-000002,WS-001,2025-03-14 01:00:00,12,2025-03-13 13:00:00,-16.4,0.0,13.6,clear,0,heavy_winter_coat_gloves_hat,-12.5,0.0,4.7,clear,0,heavy_winter_coat_gloves_hat,-3.9,0.0,8.9,1,1,1
2,FCST-000003,WS-001,2025-03-14 02:00:00,1,2025-03-14 01:00:00,-11.0,0.0,4.2,clear,0,heavy_winter_coat_gloves_hat,-11.2,0.0,4.6,clear,0,heavy_winter_coat_gloves_hat,0.2,0.0,-0.4,1,1,1
3,FCST-000004,WS-001,2025-03-14 03:00:00,24,2025-03-13 03:00:00,-6.1,0.9,0.0,fog,0,light_jacket_or_sweater,-9.7,0.0,22.1,clear,0,heavy_winter_coat_gloves_hat,3.6,0.9,-22.1,1,0,0
4,FCST-000005,WS-001,2025-03-14 04:00:00,6,2025-03-13 22:00:00,-8.5,0.5,1.8,clear,0,winter_coat_scarf_gloves,-7.7,0.0,2.6,clear,0,winter_coat_scarf_gloves,-0.8,0.5,-0.8,1,1,1


In [3]:
data = df.copy()

data["target_timestamp"] = pd.to_datetime(data["target_timestamp"])
data["forecast_issued_at"] = pd.to_datetime(data["forecast_issued_at"])

data["target_hour"] = data["target_timestamp"].dt.hour
data["target_month"] = data["target_timestamp"].dt.month
data["issue_hour"] = data["forecast_issued_at"].dt.hour
data["issue_month"] = data["forecast_issued_at"].dt.month

# Basic engineered weather indicators
data["is_rainy_forecast"] = (data["forecasted_precip_mm"] > 0.0).astype(int)
data["is_windy_forecast"] = (data["forecasted_wind_kmh"] >= 20.0).astype(int)
data["is_cold"] = (data["forecasted_temp_c"] <= 12.0).astype(int)
data["is_hot"] = (data["forecasted_temp_c"] >= 28.0).astype(int)

# Interaction features
data["precip_wind_interaction"] = data["forecasted_precip_mm"] * data["forecasted_wind_kmh"]
data["temp_wind_interaction"] = data["forecasted_temp_c"] * data["forecasted_wind_kmh"]

# Lead time as bucket
data["lead_time_bucket"] = pd.cut(
    data["lead_time_hours"],
    bins=[0, 1, 3, 6, 12, 24],
    labels=["h1", "h3", "h6", "h12", "h24"],
    include_lowest=True,
)

# Time of day buckets
def hour_bucket(hour):
    if 5 <= hour < 11:
        return "morning"
    elif 11 <= hour < 17:
        return "midday"
    elif 17 <= hour < 22:
        return "evening"
    else:
        return "night"

data["target_period"] = data["target_hour"].apply(hour_bucket)
data["issue_period"] = data["issue_hour"].apply(hour_bucket)

data.head()

,forecast_id,station_id,target_timestamp,lead_time_hours,forecast_issued_at,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_condition,forecasted_umbrella,forecasted_clothing,actual_temp_c,actual_precip_mm,actual_wind_kmh,actual_condition,actual_umbrella,actual_clothing,temp_error_c,precip_error_mm,wind_error_kmh,umbrella_correct,clothing_correct,condition_correct,target_hour,target_month,issue_hour,issue_month,is_rainy_forecast,is_windy_forecast,is_cold,is_hot,precip_wind_interaction,temp_wind_interaction,lead_time_bucket,target_period,issue_period
0,FCST-000001,WS-001,2025-03-14 00:00:00,3,2025-03-13 21:00:00,-11.3,0.0,12.6,clear,0,heavy_winter_coat_gloves_hat,-12.1,0.0,12.0,clear,0,heavy_winter_coat_gloves_hat,0.8,0.0,0.6,1,1,1,0,3,21,3,0,0,1,0,0.0,-142.38,h3,night,evening
1,FCST-000002,WS-001,2025-03-14 01:00:00,12,2025-03-13 13:00:00,-16.4,0.0,13.6,clear,0,heavy_winter_coat_gloves_hat,-12.5,0.0,4.7,clear,0,heavy_winter_coat_gloves_hat,-3.9,0.0,8.9,1,1,1,1,3,13,3,0,0,1,0,0.0,-223.04,h12,night,midday
2,FCST-000003,WS-001,2025-03-14 02:00:00,1,2025-03-14 01:00:00,-11.0,0.0,4.2,clear,0,heavy_winter_coat_gloves_hat,-11.2,0.0,4.6,clear,0,heavy_winter_coat_gloves_hat,0.2,0.0,-0.4,1,1,1,2,3,1,3,0,0,1,0,0.0,-46.20,h1,night,night
3,FCST-000004,WS-001,2025-03-14 03:00:00,24,2025-03-13 03:00:00,-6.1,0.9,0.0,fog,0,light_jacket_or_sweater,-9.7,0.0,22.1,clear,0,heavy_winter_coat_gloves_hat,3.6,0.9,-22.1,1,0,0,3,3,3,3,1,0,1,0,0.0,-0.00,h24,night,night
4,FCST-000005,WS-001,2025-03-14 04:00:00,6,2025-03-13 22:00:00,-8.5,0.5,1.8,clear,0,winter_coat_scarf_gloves,-7.7,0.0,2.6,clear,0,winter_coat_scarf_gloves,-0.8,0.5,-0.8,1,1,1,4,3,22,3,1,0,1,0,0.9,-15.30,h6,night,night


In [4]:
#Modeli eğitirken sadece o anda elimizde bulunabilecek feature’ları kullanacağız. Yani forecast var, actual yok
feature_cols = [
    "lead_time_hours",
    "forecasted_temp_c",
    "forecasted_precip_mm",
    "forecasted_wind_kmh",
    "forecasted_umbrella",
    "forecasted_clothing",
    "target_hour",
    "target_month",
    "issue_hour",
    "issue_month",
    "is_rainy_forecast",
    "is_windy_forecast",
    "is_cold",
    "is_hot",
    "precip_wind_interaction",
    "temp_wind_interaction",
    "lead_time_bucket",
    "target_period",
    "issue_period",
]

categorical_features = [
    "forecasted_clothing",
    "lead_time_bucket",
    "target_period",
    "issue_period",
]

numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X = data[feature_cols]
X.head()


,lead_time_hours,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_umbrella,forecasted_clothing,target_hour,target_month,issue_hour,issue_month,is_rainy_forecast,is_windy_forecast,is_cold,is_hot,precip_wind_interaction,temp_wind_interaction,lead_time_bucket,target_period,issue_period
0,3,-11.3,0.0,12.6,0,heavy_winter_coat_gloves_hat,0,3,21,3,0,0,1,0,0.0,-142.38,h3,night,evening
1,12,-16.4,0.0,13.6,0,heavy_winter_coat_gloves_hat,1,3,13,3,0,0,1,0,0.0,-223.04,h12,night,midday
2,1,-11.0,0.0,4.2,0,heavy_winter_coat_gloves_hat,2,3,1,3,0,0,1,0,0.0,-46.20,h1,night,night
3,24,-6.1,0.9,0.0,0,light_jacket_or_sweater,3,3,3,3,1,0,1,0,0.0,-0.00,h24,night,night
4,6,-8.5,0.5,1.8,0,winter_coat_scarf_gloves,4,3,22,3,1,0,1,0,0.9,-15.30,h6,night,night


In [5]:
def train_confidence_model(dataframe, feature_cols, target_col, threshold=0.5):
    X = dataframe[feature_cols]
    y = dataframe[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    base_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            min_samples_split=4,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
        )),
    ])

    calibrated_model = CalibratedClassifierCV(
        estimator=base_model,
        method="isotonic",
        cv=3,
    )

    calibrated_model.fit(X_train, y_train)

    #test_proba: 0 ile 1 arasında confidence
    proba = calibrated_model.predict_proba(X_test)[:, 1]
    #test_pred: bunu threshold'a göre sınıfa çevrilmiş hali
    pred = (proba >= threshold).astype(int)

    metrics = {
        "target": target_col,
        "base_rate": float(y.mean()),
        "accuracy": float(accuracy_score(y_test, pred)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "brier": float(brier_score_loss(y_test, proba)),
    }

    return calibrated_model, metrics, X_test, y_test, proba

In [6]:
#Her biri için: ayrı model eğitiliyor, ayrı metric çıkarılıyor
targets = ["umbrella_correct", "clothing_correct", "condition_correct"]

task_models = {}
task_metrics = []
task_test_outputs = {}

for target in targets:
    model, metrics, X_test, y_test, proba = train_confidence_model(data, feature_cols, target)
    task_models[target] = model
    task_metrics.append(metrics)
    task_test_outputs[target] = {
        "X_test": X_test,
        "y_test": y_test,
        "proba": proba,
    }

metrics_df = pd.DataFrame(task_metrics).sort_values("roc_auc", ascending=False)
metrics_df

,target,base_rate,accuracy,roc_auc,brier
0,umbrella_correct,0.941875,0.948958,0.926073,0.038801
2,condition_correct,0.858542,0.869792,0.904019,0.081389
1,clothing_correct,0.728125,0.776042,0.824158,0.145293


In [7]:
for target in targets:
    proba = task_test_outputs[target]["proba"]
    print(f"\n{target}")
    print(pd.Series(proba).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))


umbrella_correct
count    960.000000
mean       0.941579
std        0.133015
min        0.147081
10%        0.848469
25%        0.966293
50%        1.000000
75%        1.000000
90%        1.000000
max        1.000000
dtype: float64

clothing_correct
count    960.000000
mean       0.725182
std        0.222102
min        0.011905
10%        0.418692
25%        0.584946
50%        0.774281
75%        0.905006
90%        0.983915
max        1.000000
dtype: float64

condition_correct
count    960.000000
mean       0.858424
std        0.203533
min        0.094444
10%        0.524655
25%        0.770380
50%        0.996630
75%        0.999083
90%        0.999083
max        0.999083
dtype: float64


In [8]:
#gerçek fonksiyon hali
def predict_confidence(forecast_input: dict):
    sample = pd.DataFrame([forecast_input])


    umbrella_conf = float(task_models["umbrella_correct"].predict_proba(sample)[0, 1])
    clothing_conf = float(task_models["clothing_correct"].predict_proba(sample)[0, 1])
    condition_conf = float(task_models["condition_correct"].predict_proba(sample)[0, 1])

    return {
        "umbrella_confidence": round(umbrella_conf, 4),
        "clothing_confidence": round(clothing_conf, 4),
        "condition_confidence": round(condition_conf, 4),
    }

In [9]:
def confidence_label(score: float) -> str:
    if score >= 0.75:
        return "high"
    elif score >= 0.50:
        return "medium"
    else:
        return "low"


def enrich_confidence_output(pred: dict) -> dict:
    enriched = {}
    for key, value in pred.items():
        enriched[key] = {
            "score": round(float(value), 4),
            "label": confidence_label(float(value)),
        }
    return enriched

In [10]:
sample_1 = {
    "lead_time_hours": 1,
    "forecasted_temp_c": 18.0,
    "forecasted_precip_mm": 0.6,
    "forecasted_wind_kmh": 14.0,
    "forecasted_umbrella": 1,
    "forecasted_clothing": "light_jacket_or_sweater",
    "target_hour": 17,
    "target_month": 4,
    "issue_hour": 16,
    "issue_month": 4,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 0,
    "precip_wind_interaction": 0.6 * 14.0,
    "temp_wind_interaction": 18.0 * 14.0,
    "lead_time_bucket": "h1",
    "target_period": "evening",
    "issue_period": "evening",
}

sample_2 = {
    "lead_time_hours": 24,
    "forecasted_temp_c": 28.0,
    "forecasted_precip_mm": 0.0,
    "forecasted_wind_kmh": 10.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "very_light_clothing",
    "target_hour": 14,
    "target_month": 7,
    "issue_hour": 14,
    "issue_month": 7,
    "is_rainy_forecast": 0,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 1,
    "precip_wind_interaction": 0.0 * 10.0,
    "temp_wind_interaction": 28.0 * 10.0,
    "lead_time_bucket": "h24",
    "target_period": "midday",
    "issue_period": "midday",
}

sample_3 = {
    "lead_time_hours": 24,
    "forecasted_temp_c": 11.0,
    "forecasted_precip_mm": 5.2,
    "forecasted_wind_kmh": 32.0,
    "forecasted_umbrella": 1,
    "forecasted_clothing": "heavy_winter_coat_gloves_hat",
    "target_hour": 9,
    "target_month": 1,
    "issue_hour": 9,
    "issue_month": 1,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 1,
    "is_cold": 1,
    "is_hot": 0,
    "precip_wind_interaction": 5.2 * 32.0,
    "temp_wind_interaction": 11.0 * 32.0,
    "lead_time_bucket": "h24",
    "target_period": "morning",
    "issue_period": "morning",
}

sample_4 = {
    "lead_time_hours": 6,
    "forecasted_temp_c": 20.5,
    "forecasted_precip_mm": 0.1,
    "forecasted_wind_kmh": 18.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "light_jacket_or_sweater",
    "target_hour": 20,
    "target_month": 5,
    "issue_hour": 14,
    "issue_month": 5,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 0,
    "precip_wind_interaction": 0.1 * 18.0,
    "temp_wind_interaction": 20.5 * 18.0,
    "lead_time_bucket": "h6",
    "target_period": "evening",
    "issue_period": "midday",
}

sample_5 = {
    "lead_time_hours": 3,
    "forecasted_temp_c": 31.0,
    "forecasted_precip_mm": 0.0,
    "forecasted_wind_kmh": 8.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "very_light_clothing",
    "target_hour": 15,
    "target_month": 8,
    "issue_hour": 12,
    "issue_month": 8,
    "is_rainy_forecast": 0,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 1,
    "precip_wind_interaction": 0.0 * 8.0,
    "temp_wind_interaction": 31.0 * 8.0,
    "lead_time_bucket": "h3",
    "target_period": "midday",
    "issue_period": "midday",
}

samples = [sample_1, sample_2, sample_3, sample_4, sample_5]

for i, sample in enumerate(samples, start=1):
    raw_pred = predict_confidence(sample)
    enriched = enrich_confidence_output(raw_pred)
    print(f"--- Sample {i} ---")
    print(enriched)
    print()

--- Sample 1 ---
{'umbrella_confidence': {'score': 1.0, 'label': 'high'}, 'clothing_confidence': {'score': 0.9074, 'label': 'high'}, 'condition_confidence': {'score': 0.9966, 'label': 'high'}}

--- Sample 2 ---
{'umbrella_confidence': {'score': 0.8621, 'label': 'high'}, 'clothing_confidence': {'score': 0.4301, 'label': 'low'}, 'condition_confidence': {'score': 0.4054, 'label': 'low'}}

--- Sample 3 ---
{'umbrella_confidence': {'score': 0.625, 'label': 'medium'}, 'clothing_confidence': {'score': 0.6034, 'label': 'medium'}, 'condition_confidence': {'score': 0.5291, 'label': 'medium'}}

--- Sample 4 ---
{'umbrella_confidence': {'score': 0.9958, 'label': 'high'}, 'clothing_confidence': {'score': 0.8684, 'label': 'high'}, 'condition_confidence': {'score': 0.998, 'label': 'high'}}

--- Sample 5 ---
{'umbrella_confidence': {'score': 1.0, 'label': 'high'}, 'clothing_confidence': {'score': 0.8739, 'label': 'high'}, 'condition_confidence': {'score': 0.998, 'label': 'high'}}

